# Probando DarkELF

In [1]:
import sys
from pathlib import Path

import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt
import pandas as pd

# Point Python at the local DarkELF package.
darkelf_root = Path("/home/lurishi/DarkELF")
sys.path.append(str(darkelf_root))
alpha_em = 1 / 137.035999084
from darkelf import darkelf
%matplotlib qt

custom_params = {
    'font.family': 'serif',
}

plt.rcParams.update(custom_params)
# Pick the target material whose dielectric response you want to use.
# Ge and Si are the default tabulated materials in DarkELF.
target = "Si"
material = darkelf(
    target=target,
    filename=f"{target}_gpaw_withLFE.dat",
    phonon_filename=f"{target}_epsphonon_data2K.dat",
)


 .... Loading files for Si
Loaded Si_gpaw_withLFE.dat for epsilon in electron regime
electronic ELF taken or calculated from J. Enkovaara et al.,Electronic structure calculations with GPAW: a real-space implementation of the projector augmented-wave method,Journal of Physics:Condensed Matter22(2010) 253202.
Warning! eps for phonon frequencies not loaded. Need to set phonon_filename to perform data-driven, single phonon calculations
Warning, Si_eps_electron_opticallimit.dat does not exist! dielectric function in optical limit not loaded. Needed for absorption calculations in superconductors.
Zion(k) for Migdal calculation taken or calculated from: P. J. Brown, A. G. Fox, E. N. Maslen, M. A. OKeefe,and B. T. M. Willis, “Intensity of diffracted intensities,” in International Tables for Crystallography (American Cancer Society, 2006) Chap. 6.1, pp. 554–595, https://onlinelibrary.wiley.com/doi/pdf/10.1107/97809553602060000
Loaded /home/lurishi/DarkELF/darkelf/../data/Si/Si_pDoS.dat for part

In [2]:
# This evaluates the integrand in the displayed formula:
#   1/k * Im[-1/epsilon(omega, k)]
#   + k * (beta^2 - omega^2/k^2) * Im[1/(-k^2 + epsilon(omega, k) * omega^2)]
# DarkELF provides epsilon1/epsilon2 and elf; the second term is built directly here.
def mcp_integrand(k, omega, beta, material, method="grid"):
    eps1 = material.eps1(omega, k, method=method)
    eps2 = material.eps2(omega, k, method=method)
    epsilon = eps1 + 1j * eps2

    with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        longitudinal = material.elf(omega, k, method=method) / k  # Im[-1/epsilon] = ELF

        omega_over_k = omega / k
        k2 = k * k
        omega2 = omega * omega
        beta2 = beta * beta
        denom = -k2 + epsilon * omega2
        transverse = k * (beta2 - omega_over_k * omega_over_k) * np.imag(1.0 / denom)

    total = longitudinal + transverse
    if not np.isfinite(total):
        return 0.0
    return total  # pensar las unidades de esto


def mcp_integral_scipy(omega, beta, material, method="grid", kmin=1e-6, kmax=None):
    if kmax is None:
        kmax = material.kmax

    value, error = integrate.quad(
        lambda k: mcp_integrand(k, omega, beta, material, method=method),
        kmin,
        kmax,
        limit=200,
    )
    return value, error

In [3]:
def mcp_integral(omega, beta, material, method="grid", kmin=1e-6, kmax=None, npts=1000, k_scale=None):

    k_values = np.logspace(np.log10(omega/beta), 3, npts)
    integrand_values = np.array([
        mcp_integrand(k, omega, beta, material, method=method)
        for k in k_values
    ])

    value = np.trapezoid(integrand_values, k_values)
    return value

In [4]:
beta = 1
omega_values = np.linspace(5, 50, 1000)
dsigma_dE = np.zeros_like(omega_values)

for i, omega in enumerate(omega_values):
    value = mcp_integral(omega, beta, material, method="grid", kmax=None, npts=400)
    #value,error= mcp_integral_scipy(omega, beta, material, kmin=1e-6, kmax=None)
    dsigma_dE[i] = value

In [5]:
dsigma_dE

array([-1.34097855e-01, -1.91025784e-01, -2.41743396e-01, -2.42064482e-01,
       -2.42873436e-01, -2.44362882e-01, -2.46780770e-01, -2.50414452e-01,
       -2.55550711e-01, -2.90909160e-01, -3.38278495e-01, -3.74832459e-01,
       -4.03845729e-01, -4.27394961e-01, -4.46861983e-01, -4.63202735e-01,
       -4.09485027e-01, -3.40135513e-01, -2.70724035e-01, -2.05382955e-01,
       -1.48357940e-01, -1.02530675e-01, -6.83701564e-02, -5.50418254e-02,
       -4.07595169e-02, -2.53423290e-02, -8.66180107e-03,  9.42630732e-03,
        2.90812517e-02,  5.04259418e-02,  6.55131201e-02,  8.08441388e-02,
        9.63868848e-02,  1.12101748e-01,  1.27939860e-01,  1.43849941e-01,
        1.59769852e-01,  1.77118412e-01,  1.94881677e-01,  2.12702904e-01,
        2.30365294e-01,  2.47575889e-01,  2.63958648e-01,  2.79032377e-01,
        2.93976970e-01,  3.10894088e-01,  3.29431509e-01,  3.49808812e-01,
        3.72281637e-01,  3.97150997e-01,  4.24771705e-01,  4.52952424e-01,
        4.66041227e-01,  

In [6]:
plt.close('all')
alpha_em = 1 / 137.035999084
n = 5e22
dsigma_dE_scaled = dsigma_dE *(2*alpha_em*(1e-4)**2)/(n*np.pi*beta**2)

plt.plot(omega_values, dsigma_dE_scaled)
plt.xlabel("Energy (eV)")
plt.ylabel("Normalized dσ/dE")

plt.grid(linestyle="--")

In [7]:
import pandas as pd
df = pd.read_csv("/home/lurishi/Escritorio/Doctorado/Mcp_gráficos_papers/Darkelfmcp.csv",sep=";")


In [8]:
df.dtypes

Er           float64
dsigma_de    float64
dtype: object

In [9]:
plt.plot(df[df.columns[0]], df[df.columns[1]]/df[df.columns[1]].max())
plt.plot(omega_values, dsigma_dE_scaled/dsigma_dE_scaled.max())
plt.xlabel("Energy (eV)")
plt.ylabel("Normalized dσ/dE")
plt.legend([f"Paper Rouven Essig", "Mi simulación"])
plt.grid(linestyle="--")


In [ ]:
plt.close('all')
factor = 4.6*1e4
plt.figure(figsize=(8,6))
plt.plot(df[df.columns[0]], df[df.columns[1]])
plt.plot(omega_values, dsigma_dE_scaled*factor)
plt.xlabel("Energy (eV)")
plt.ylabel("Normalized dσ/dE")
plt.legend([f"Paper Rouven Essig", "Mi simulación"])
plt.grid(linestyle="--")

qt.qpa.wayland.textinput: virtual void QtWaylandClient::QWaylandTextInputv3::zwp_text_input_v3_leave(wl_surface*) Got leave event for surface 0x0 with focusing surface 0x5ab2223586e0
qt.qpa.wayland: There are no outputs - creating placeholder screen
